In [2]:
from dotenv import load_dotenv

MODEL = "claude-haiku-4-5"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="google/embeddinggemma-300m")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()

In [ ]:
# retriever.invoke("Who is Avery?")

[Document(id='72a1c07c-126e-430b-8dc2-7eeb1a6548e8', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership during Insurellm's foundational year led to successful product launches and securing initial funding.  \n\n- **2016**: **Meets Expectations**  \n  Growth continued, though challenges arose in operational efficiency that required Avery's attention.  \n\n- **2017**: **Developing**  \n  Market competition intensified, and monthly sales metrics were below targets. Avery implemented new strategies which required a steep learning curve.  \n\n- **2018**: **Exceeds Expe

In [ ]:
# from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(temperature=0, model_name=MODEL)
# llm.invoke("Who is Avery?")

AIMessage(content='I don\'t have enough context to answer your question about who Avery is. "Avery" could refer to many people, including:\n\n- A fictional character from a book, movie, or TV show\n- A real person you know personally\n- A public figure or celebrity\n- A historical figure\n- A character from a game or other media\n\nCould you provide more details about which Avery you\'re asking about? That would help me give you a more useful answer.', additional_kwargs={}, response_metadata={'id': 'msg_011CczETjpdnWv7fbMmXEoiA', 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 13, 'output_tokens': 104, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'stop_details': None, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anth

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

In [ ]:
import gradio as gr

gr.ChatInterface(answer_question).launch()